# MNIST Diffusion — Passive, Active & Linear (learn M) + FID

Runs the full pipeline:
1. Train **Passive**, **Active**, and **Linear (learnable M, fixed-norm constraint)** diffusion models
2. Save real MNIST test-set PNGs and precompute Inception statistics (once)
3. Generate samples from each model
4. Compute **pytorch-fid** scores

Training cells run sequentially. For parallel training across 3 GPUs, see the `%%bash` cell at the bottom of the training section.

## Setup

In [ ]:
import os, torch, subprocess, sys

%cd /home/agnish/Documents/Projects/Final_Diffusion_Paper_code/MNIST

n_gpus = torch.cuda.device_count()
print(f"GPUs: {n_gpus}")
for i in range(n_gpus):
    print(f"  [{i}] {torch.cuda.get_device_name(i)}")

os.makedirs("results",  exist_ok=True)
os.makedirs("fid_data", exist_ok=True)

---
## Training

Each model trains for 60 epochs, saving a checkpoint every 5 epochs to `results/`.

| Model | Checkpoint |
|---|---|
| Passive | `results/Passive_60_T1.0_k1.0.pt` |
| Active  | `results/Active_60_Tp0.001_Ta1.0_tau0.5.pt` |
| Linear (learn M, fixed norm) | `results/Linear_60_learnM_fixed_norm.pt` |

Run cells individually (sequential) **or** use the `%%bash` parallel cell below.

### Passive Diffusion

SDE: `dx = -k x dt + √(2T) dW`  
`T=1.0`, `k=1.0`, `total_time=2.0`

In [ ]:
!CUDA_VISIBLE_DEVICES=0 python train_mnist_fixed_normM.py \
    --model_type passive \
    --T 1.0 \
    --k 1.0 \
    --total_time 2.0 \
    --epochs 60 \
    --batch_size 128 \
    --lr 0.001 \
    --timesteps 1000 \
    --model_base_dim 64 \
    --save_freq 5 \
    --n_samples 36

### Active Diffusion

SDE: `dx = (-k x + η) dt + √(2Tp) dW_x`,  `dη = -(1/τ) η dt + √(2Ta)/τ dW_η`  
`Tp=0.001`, `Ta=1.0`, `τ=0.5`, `k=1.0`

In [ ]:
!CUDA_VISIBLE_DEVICES=1 python train_mnist_fixed_normM.py \
    --model_type active \
    --Tp 0.001 \
    --Ta 1.0 \
    --tau 0.5 \
    --k 1.0 \
    --total_time 2.0 \
    --epochs 60 \
    --batch_size 128 \
    --lr 0.001 \
    --timesteps 1000 \
    --model_base_dim 64 \
    --save_freq 5 \
    --n_samples 36

### Linear Diffusion — Learnable M (fixed Frobenius norm)

SDE: `dz = M z dt + D dW`,  `z = (x, η)`  
M and D initialised to the active-matter special case (`k=1`, `τ=0.5`, `Ta=1`).  
`--learn_M --M_constraint fixed_norm` — only the **direction** of M is learned; its Frobenius norm is held constant.

In [ ]:
!CUDA_VISIBLE_DEVICES=2 python train_mnist_fixed_normM.py \
    --model_type linear \
    --M -1.0 1.0 0.0 -2.0 \
    --D 0.0 0.0 0.0 2.828 \
    --learn_M \
    --M_constraint fixed_norm \
    --M_lr_factor 0.01 \
    --stability_weight 10.0 \
    --M_l2_weight 0.01 \
    --eta0_mode stationary_marginal \
    --total_time 2.0 \
    --epochs 60 \
    --batch_size 128 \
    --lr 0.001 \
    --timesteps 1000 \
    --model_base_dim 64 \
    --save_freq 5 \
    --n_samples 36 \
    --name_suffix learnM_fixed_norm

### (Alternative) All 3 models in parallel — one cell, 3 GPUs

Skip the three cells above and run this instead if you have ≥ 3 GPUs and want simultaneous training.  
Logs are written to `results/*.log`; tail them in a terminal with `tail -f results/passive.log`.

In [ ]:
%%bash
CUDA_VISIBLE_DEVICES=0 python train_mnist_fixed_normM.py \
    --model_type passive \
    --T 1.0 --k 1.0 --total_time 2.0 \
    --epochs 60 --batch_size 128 --lr 0.001 \
    --timesteps 1000 --model_base_dim 64 --save_freq 5 --n_samples 36 \
    > results/passive.log 2>&1 &

CUDA_VISIBLE_DEVICES=1 python train_mnist_fixed_normM.py \
    --model_type active \
    --Tp 0.001 --Ta 1.0 --tau 0.5 --k 1.0 --total_time 2.0 \
    --epochs 60 --batch_size 128 --lr 0.001 \
    --timesteps 1000 --model_base_dim 64 --save_freq 5 --n_samples 36 \
    > results/active.log 2>&1 &

CUDA_VISIBLE_DEVICES=2 python train_mnist_fixed_normM.py \
    --model_type linear \
    --M -1.0 1.0 0.0 -2.0 --D 0.0 0.0 0.0 2.828 \
    --learn_M --M_constraint fixed_norm \
    --M_lr_factor 0.01 --stability_weight 10.0 --M_l2_weight 0.01 \
    --eta0_mode stationary_marginal --total_time 2.0 \
    --epochs 60 --batch_size 128 --lr 0.001 \
    --timesteps 1000 --model_base_dim 64 --save_freq 5 --n_samples 36 \
    --name_suffix learnM_fixed_norm \
    > results/linear_learnM.log 2>&1 &

wait
echo "All training jobs finished."

---
## FID — pytorch-fid

Four steps:
1. Save real MNIST test images as PNGs (once)
2. Precompute Inception statistics over real images (once)
3. Generate samples from each model
4. Compute FID for each model

In [ ]:
# Install pytorch-fid if needed
!pip install pytorch-fid -q

### Step 1 — Save real MNIST test-set PNGs  *(run once)*

In [ ]:
!python save_real_mnist.py --output_dir fid_data/real

### Step 2 — Precompute real Inception statistics  *(run once)*

In [ ]:
!python -m pytorch_fid --save-stats fid_data/real fid_data/real_stats.npz --device cuda:0

### Step 3 — Generate samples from each model

In [ ]:
# Passive
!CUDA_VISIBLE_DEVICES=0 python generate_fid_samples.py \
    --ckpt results/Passive_60_T1.0_k1.0.pt \
    --output_dir fid_data/passive_60 \
    --n_samples 10000 \
    --sample_batch_size 256

In [ ]:
# Active
!CUDA_VISIBLE_DEVICES=0 python generate_fid_samples.py \
    --ckpt results/Active_60_Tp0.001_Ta1.0_tau0.5.pt \
    --output_dir fid_data/active_60 \
    --n_samples 10000 \
    --sample_batch_size 256

In [ ]:
# Linear — learnable M, fixed norm
!CUDA_VISIBLE_DEVICES=0 python generate_fid_samples.py \
    --ckpt results/Linear_60_learnM_fixed_norm.pt \
    --output_dir fid_data/linear_learnM_60 \
    --n_samples 10000 \
    --sample_batch_size 256

*(Optional)* Generate all 3 in parallel across GPUs:

In [ ]:
%%bash
CUDA_VISIBLE_DEVICES=0 python generate_fid_samples.py \
    --ckpt results/Passive_60_T1.0_k1.0.pt \
    --output_dir fid_data/passive_60 \
    --n_samples 10000 --sample_batch_size 256 &

CUDA_VISIBLE_DEVICES=1 python generate_fid_samples.py \
    --ckpt results/Active_60_Tp0.001_Ta1.0_tau0.5.pt \
    --output_dir fid_data/active_60 \
    --n_samples 10000 --sample_batch_size 256 &

CUDA_VISIBLE_DEVICES=2 python generate_fid_samples.py \
    --ckpt results/Linear_60_learnM_fixed_norm.pt \
    --output_dir fid_data/linear_learnM_60 \
    --n_samples 10000 --sample_batch_size 256 &

wait
echo "All sample generation jobs finished."

### Step 4 — Compute FID scores

In [ ]:
# Capture FID output into Python variables for a clean summary
def compute_fid(sample_dir, stats_npz="fid_data/real_stats.npz", device="cuda:0"):
    result = subprocess.run(
        [sys.executable, "-m", "pytorch_fid", stats_npz, sample_dir, "--device", device],
        capture_output=True, text=True
    )
    print(result.stdout.strip())
    if result.returncode != 0:
        print(result.stderr.strip())
        return None
    # pytorch-fid prints: "FID:  <value>"
    for line in result.stdout.splitlines():
        if "FID" in line:
            return float(line.split()[-1])
    return None

print("=" * 40)
print("Passive")
passive_fid = compute_fid("fid_data/passive_60")

print("\nActive")
active_fid = compute_fid("fid_data/active_60")

print("\nLinear (learn M, fixed norm)")
linear_fid = compute_fid("fid_data/linear_learnM_60")

print("\n" + "=" * 40)
print(f"  Passive FID             : {passive_fid}")
print(f"  Active  FID             : {active_fid}")
print(f"  Linear (learn M) FID    : {linear_fid}")
print("=" * 40)

with open("fid_results.txt", "w") as f:
    f.write(f"Passive FID          : {passive_fid}\n")
    f.write(f"Active  FID          : {active_fid}\n")
    f.write(f"Linear (learn M) FID : {linear_fid}\n")
print("Results written to fid_results.txt")

---
## FID vs Epoch — multi-checkpoint sweep

Edit `EPOCHS` to the checkpoints you want to evaluate.  
For each epoch the cell will:
1. Generate samples from all 3 models **in parallel** (one subprocess per GPU)
2. Compute FID against the precomputed real-image stats
3. Collect results into a printed table, a CSV, and a plot

**Prerequisite:** `fid_data/real_stats.npz` must exist (Steps 1 & 2 of the pytorch-fid section above).

In [ ]:
# ── Configuration ───────────────────────────────────────────────────────
EPOCHS = [10, 20, 30, 40, 50, 60]   # <-- edit this list

N_SAMPLES    = 10_000
SAMPLE_BATCH = 256
REAL_STATS   = "fid_data/real_stats.npz"
FID_DEVICE   = "cuda:0"   # device used for FID Inception features

# Checkpoint path templates  (match get_checkpoint_path() in train_mnist_fixed_normM.py)
CKPT_TEMPLATES = {
    "passive": "results/Passive_{e}_T1.0_k1.0.pt",
    "active":  "results/Active_{e}_Tp0.001_Ta1.0_tau0.5.pt",
    "linear":  "results/Linear_{e}_learnM_fixed_norm.pt",
}

# GPU assignments for sample generation (one process per model, run in parallel)
GEN_GPUS = {
    "passive": "0",
    "active":  "1",
    "linear":  "2",
}
# ────────────────────────────────────────────────────────────────────────

def _run_fid(sample_dir):
    """Run pytorch-fid and return the FID score as a float, or None on error."""
    r = subprocess.run(
        [sys.executable, "-m", "pytorch_fid", REAL_STATS, sample_dir, "--device", FID_DEVICE],
        capture_output=True, text=True,
    )
    if r.returncode != 0:
        print(f"  [ERROR] pytorch-fid failed for {sample_dir}:\n{r.stderr.strip()}")
        return None
    for line in r.stdout.splitlines():
        if "FID" in line:
            return float(line.split()[-1])
    return None


def _dir_has_samples(out_dir, n):
    """Return True if out_dir already contains at least n PNG files."""
    if not os.path.isdir(out_dir):
        return False
    return sum(1 for f in os.listdir(out_dir) if f.endswith(".png")) >= n


def _generate_parallel(epoch):
    """Launch one generate_fid_samples.py process per model in parallel, then wait.
    Skips generation for any model whose output directory already has enough PNGs."""
    procs, out_dirs = {}, {}
    for model, tmpl in CKPT_TEMPLATES.items():
        ckpt = tmpl.format(e=epoch)
        if not os.path.exists(ckpt):
            print(f"  [SKIP] checkpoint not found: {ckpt}")
            continue
        out_dir = f"fid_data/{model}_{epoch}"
        if _dir_has_samples(out_dir, N_SAMPLES):
            print(f"  [REUSE] {out_dir} already has ≥{N_SAMPLES} PNGs")
            out_dirs[model] = out_dir
            continue
        os.makedirs(out_dir, exist_ok=True)
        env = os.environ.copy()
        env["CUDA_VISIBLE_DEVICES"] = GEN_GPUS[model]
        proc = subprocess.Popen(
            [
                sys.executable, "generate_fid_samples.py",
                "--ckpt", ckpt,
                "--output_dir", out_dir,
                "--n_samples", str(N_SAMPLES),
                "--sample_batch_size", str(SAMPLE_BATCH),
            ],
            env=env,
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL,
        )
        procs[model] = proc
        out_dirs[model] = out_dir
        print(f"  Launched: {model} epoch {epoch:3d} → {out_dir}")

    for model, proc in procs.items():
        proc.wait()
        if proc.returncode != 0:
            print(f"  [ERROR] generation failed for {model} epoch {epoch}")
    return out_dirs


# ── Main sweep ──────────────────────────────────────────────────────────
results = {model: {} for model in CKPT_TEMPLATES}   # results[model][epoch] = fid

for epoch in EPOCHS:
    print(f"\n{'─'*50}")
    print(f"Epoch {epoch}: generating samples …")
    out_dirs = _generate_parallel(epoch)

    print(f"Epoch {epoch}: computing FID …")
    for model, out_dir in out_dirs.items():
        fid_val = _run_fid(out_dir)
        results[model][epoch] = fid_val
        print(f"  {model:10s} epoch {epoch:3d}  →  FID = {fid_val}")

print("\n✓ Sweep complete.")

In [ ]:
# ── Results table + CSV ──────────────────────────────────────────────────
import csv

print(f"{'Epoch':>6}  {'Passive FID':>12}  {'Active FID':>12}  {'Linear FID':>12}")
print("─" * 50)
for epoch in EPOCHS:
    row = f"{epoch:>6}"
    for model in ["passive", "active", "linear"]:
        val = results[model].get(epoch)
        row += f"  {val:>12.4f}" if val is not None else f"  {'N/A':>12}"
    print(row)

with open("fid_vs_epoch.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["epoch", "passive_fid", "active_fid", "linear_fid"])
    for epoch in EPOCHS:
        w.writerow([
            epoch,
            results["passive"].get(epoch),
            results["active"].get(epoch),
            results["linear"].get(epoch),
        ])
print("\nSaved to fid_vs_epoch.csv")

In [ ]:
# ── FID vs Epoch plot ────────────────────────────────────────────────────
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 4))

labels  = {"passive": "Passive", "active": "Active", "linear": "Linear (learn M)"}
markers = {"passive": "o",       "active": "s",      "linear": "^"}

for model in ["passive", "active", "linear"]:
    xs = [e for e in EPOCHS if results[model].get(e) is not None]
    ys = [results[model][e] for e in xs]
    ax.plot(xs, ys, marker=markers[model], label=labels[model])

ax.set_xlabel("Epoch")
ax.set_ylabel("FID (pytorch-fid, Inception v3)")
ax.set_title("FID vs Epoch — MNIST Diffusion Models")
ax.legend()
ax.grid(True, linestyle="--", alpha=0.5)
fig.tight_layout()
fig.savefig("fid_vs_epoch.png", dpi=150)
plt.show()
print("Plot saved to fid_vs_epoch.png")

> **Tip:** Re-run only the sweep cell (`cell-27`) with a different `EPOCHS` list to add more checkpoints — previously generated sample directories in `fid_data/` are reused automatically (generation is skipped if the directory already exists and is non-empty).